In [2]:
import matplotlib.pyplot as plt
import numpy as np
import h5py
import copy
import pandas as pd
from scipy import optimize
from matplotlib import pyplot as plt
import pickle
from itertools import cycle
import matplotlib.colors as mcolors
import astropy.units as u
import astropy.constants as const

from photochem.utils import stars
import PICASO_Climate_grid_121625 as picaso_grid
import Photochem_grid_121625 as photochem_grid
import Reflected_Spectra_grid_13026 as Reflected_Spectra
import Reflected_Spectra_SpecificInputs as Reflected_Spectra_SpecificInputs_Script
from picaso.photochem import EquilibriumChemistry
import GraphsKey

import os
from pathlib import Path

current_directory = Path.cwd()
references_directory_path = "Installation&Setup_Instructions/picasofiles/reference"
PYSYN_directory_path = "Installation&Setup_Instructions/picasofiles/grp/redcat/trds"
print(os.path.join(current_directory, references_directory_path))
print(os.path.join(current_directory, PYSYN_directory_path))

os.environ['picaso_refdata']= os.path.join(current_directory, references_directory_path)
os.environ['PYSYN_CDBS']= os.path.join(current_directory, PYSYN_directory_path)


import picaso.justdoit as jdi
import picaso.justplotit as jpi

/mmfs1/gscratch/stf/elizap/MiniNeptuneGrid26_PostBac/Installation&Setup_Instructions/picasofiles/reference
/mmfs1/gscratch/stf/elizap/MiniNeptuneGrid26_PostBac/Installation&Setup_Instructions/picasofiles/grp/redcat/trds
/mmfs1/gscratch/stf/elizap/MiniNeptuneGrid26_PostBac/Installation&Setup_Instructions/picasofiles/reference
/mmfs1/gscratch/stf/elizap/MiniNeptuneGrid26_PostBac/Installation&Setup_Instructions/picasofiles/grp/redcat/trds
/mmfs1/gscratch/stf/elizap/MiniNeptuneGrid26_PostBac/Installation&Setup_Instructions/picasofiles/reference/opacities/opacities_photochem_0.1_250.0_R15000.db
/mmfs1/gscratch/stf/elizap/MiniNeptuneGrid26_PostBac/Installation&Setup_Instructions/picasofiles/reference
/mmfs1/gscratch/stf/elizap/MiniNeptuneGrid26_PostBac/Installation&Setup_Instructions/picasofiles/grp/redcat/trds
/mmfs1/gscratch/stf/elizap/MiniNeptuneGrid26_PostBac/Installation&Setup_Instructions/picasofiles/reference
/mmfs1/gscratch/stf/elizap/MiniNeptuneGrid26_PostBac/Installation&Setup_Inst

In [3]:
def find_pbot(sol=None, solaer=None, tol=0.9):

    """
    Parameters:
    pressures: ndarray
        Pressure at each atmospheric layer in dynes/cm^2
    H2Oaer: ndarray
        Mixing ratio of H2O aerosols.
    tol: float, optional
        The threshold value for which we define the beginning of the cloud, 
        by default 1e-4. 

    Returns:
    P_bottom: float
        The cloud bottom pressure in dynes/cm^2
        
    """

    pressure = sol['pressure']
    H2Oaer = solaer['H2Oaer']

    # There is no water cloud in the model, so we return None
    # For the cloud bottom of pressure

    if np.max(H2Oaer) < 1e-20:
        return None

    # Normalize so that max value is 1
    H2Oaer_normalized = H2Oaer/np.max(H2Oaer)

    # loop from bottom to top of atmosphere, cloud bottom pressure
    # defined as the index level where the normalized cloud mixing ratio
    # exeeds tol .

    ind = None
    
    for i, val in enumerate(H2Oaer_normalized):
        if val > tol:
            ind = i
            break

    if ind is None:
        raise Exception('A problem happened when trying to find the bottom of the cloud.')

    # Bottom of the cloud
    pbot = pressure[ind]

    return pbot

In [4]:
def reflected_spectrum_planet_Sun(rad_plan=None, planet_metal=None, tint=None, semi_major=None, ctoO=None, Kzz=None, phase_angle=None, sol_path=None, soleq_path=None, Photochem_file=False,atmosphere_kwargs={}, outputfile=None):

    """
    This finds the reflected spectra of a planet similar to K218b around a Sun.
start_case.inputs['atmosphere']['exclude_mol'] = {'CH4': 0}
    Parameters:
    rad_plan: float
        This is the radius of the planet in units of Earth radii.
    planet_metal: float
        This is the planet's metallicity in units of log10 x Solar metallicity.
    tint: float
        This is the planet's internal temperature in Kelvin.
    semi_major: float
        This is the semi major axis of your planet's orbit in units of AU.
    ctoO: float
        This is the carbon to oxygen ratio of your planet's atmosphere in units of xSolar c/o.
    Kzz: float
        This is the eddy diffusion coefficient in logspace (i.e. the power of 10) in cm/s^2.
    phase_angle: float
        This is the phase of orbit the planet is in relative to its star and the observer (i.e. how illuminated it is), units of radians.
    Photochem_file: string
        This is the path to the Photochem grid you would like to pull composition information from.
    atmosphere_kwargs: dict 'exclude_mol': value where value is a string
        If left empty, all molecules are included, but can limit how many molecules are calculated. 

    Results: IDK for sure though
    wno: grid of 150 points
        ???
    fpfs: grid of 150 points
        This is the relative flux of the planet and star (fp/fs). 
    alb: grid of 150 points
        ???
    np.array(clouds): grid of 150 points
        This is a grid of whether or not a cloud was used to make the reflective spectra using the binary equivalent to booleans (True=1, False=0).
        
    """

    # Create and empty dictionary for later results
    
    current_directory = Path.cwd()
    opacity_file_path = "Installation&Setup_Instructions/picasofiles/reference/opacities/opacities_photochem_0.1_250.0_R15000.db"  
    opacity_path=os.path.join(current_directory, opacity_file_path)
    print(opacity_path)
    opacity = jdi.opannection(filename_db=opacity_path, wave_range=[0.1,2.5])

    planet_metal = float(planet_metal)
    
    start_case = jdi.inputs()

    # Then calculate the composition from the TP profile
    class planet:
        
        planet_radius = (rad_plan*6.371e+6*u.m) # in meters
        planet_mass = picaso_grid.mass_from_radius_chen_kipping_2017(R_rearth=rad_plan)*(5.972e+24) # in kg
        planet_Teq = picaso_grid.calc_Teq_SUN(distance_AU=semi_major) # Equilibrium temp (K)
        planet_grav = (const.G * (planet_mass)) / ((planet_radius)**2) # of K2-18b in m/s^2
        planet_ctoO = ctoO # in xSolar

    class Sun:
        
        stellar_radius = 1 # Solar radii
        stellar_Teff = 5778 # K
        stellar_metal = 0.0 # log10(metallicity)
        stellar_logg = 4.4 # log10(gravity), in cgs units

    solar_zenith_angle = 60 # Used in Tsai et al. (2023)
        
    # Star and Planet Parameters (Stay the Same & Should Match Photochem & PICASO)
    start_case.phase_angle(phase_angle, num_tangle=8, num_gangle=8) #radians, using default here

    jupiter_mass = const.M_jup.value # in kg
    jupiter_radius = 69911e+3 # in m
    start_case.gravity(gravity=planet.planet_grav, gravity_unit=jdi.u.Unit('m/(s**2)'), radius=(planet.planet_radius.value)/jupiter_radius, radius_unit=jdi.u.Unit('R_jup'), mass=(planet.planet_mass)/jupiter_mass, mass_unit=jdi.u.Unit('M_jup'))
    
    # star temperature, metallicity, gravity, and opacity (default opacity is opacity.db in the reference folder)
    start_case.star(opannection=opacity, temp=Sun.stellar_Teff, logg=Sun.stellar_logg, semi_major=semi_major, metal=Sun.stellar_metal, radius=Sun.stellar_radius, radius_unit=jdi.u.R_sun, semi_major_unit=jdi.u.au)

    # Match Photochemical Files
    if Photochem_file is not True:
        sol_dict, soleq_dict, PT_list, convergence_PC, convergence_TP = Reflected_Spectra.find_Photochem_match(filename=Photochem_file, rad_plan=rad_plan, log10_planet_metallicity=planet_metal, tint=tint, semi_major=semi_major, ctoO=ctoO,Kzz=Kzz)

    elif Photochem_file is True:
        with open(sol_path, 'rb') as file:
            sol_dict = pickle.load(file)
        with open(soleq_path, 'rb') as file:
            soleq_dict = pickle.load(file)
    
            
    # Determine Planet Atmosphere & Composition

    atm, sol_dict_aer = Reflected_Spectra.make_picaso_atm(sol_dict) # Converted Pressure of Photochem, in dynes/cm^2, back to bars and flip all arrays before placing into PICASO
    print(type(atm['pressure']))
    print(f'Length of pressure vs other element: {len(atm['pressure'])} vs {len(atm['He'])}')

    # Limit atmosphere to pressures 1000 bars and below.

    atm_filtered = {}
    
    # Threshold value
    threshold = 1000
    
    # Filter the list for a specific key (e.g., 'key1')
    # Convert list to NumPy array for efficient filtering
    arr = np.array(atm['temperature'])
    
    # Use boolean indexing to filter values less than the threshold
    filtered_arr = arr[arr < threshold]
    
    # Update the dictionary (optional: convert back to list)
    atm_filtered['temperature'] = filtered_arr # or keep as a NumPy array: data['key1'] = filtered_arr
    atm_filtered['pressure'] = atm['pressure'][:len(atm_filtered['temperature'])] # filter temperature by index

   # Define keys to ignore
    exclude = {'pressure', 'temperature'}
    
    for key, value in atm.items():
        if key not in exclude:
            atm_filtered[key] = value[:len(atm_filtered['temperature'])]
            #print(f'updated with {key}: {atm_filtered}')
        
    
    #print(f"Original dictionary: {atm.keys()}")
    #print(f"Filtered 'pressure' values (below {threshold}): {len(atm_filtered['pressure'])}, then temperature should be the same length as new pressure: {len(atm_filtered['temperature'])}")
    #print(f"Quick check with PT association of old and new dictionaries: filtered pres: {atm_filtered['pressure'][5]}, filtered temp: {atm_filtered['temperature'][5]}, original pres: {atm['pressure'][5]}, original temp: {atm['temperature'][5]}")

    df_atmo = jdi.pd.DataFrame(atm_filtered)

    plt.gca().invert_yaxis()
    plt.semilogy(df_atmo['temperature'], df_atmo['pressure'], ls='-', c='red') 
    plt.semilogy(atm['temperature'], atm['pressure'], ls='--', c='blue')

    # 4. Customize and display
    plt.title("Filtered PT Profile")
    plt.xlabel("Temperature in K")
    plt.ylabel("Pressure in bars")
    plt.legend() # Displays the labels
    plt.show()

    if 'exclude_mol' in atmosphere_kwargs:
        for sp in atmosphere_kwargs['exclude_mol']:
            print(f'This should show the species you are excluding: {sp}')
            
            if sp in df_atmo:
                df_atmo[sp] *= 0
                print(df_atmo[sp])
    
    start_case.atmosphere(df = df_atmo) 
    df_cldfree = start_case.spectrum(opacity, calculation='reflected', full_output=True)
    wno_cldfree, alb_cldfree, fpfs_cldfree = df_cldfree['wavenumber'], df_cldfree['albedo'], df_cldfree['fpfs_reflected']
    _, alb_cldfree_grid = jdi.mean_regrid(wno_cldfree, alb_cldfree, R=150)
    wno_cldfree_grid, fpfs_cldfree_grid = jdi.mean_regrid(wno_cldfree, fpfs_cldfree, R=150)

    print(f'This is the length of the grids created: {len(wno_cldfree_grid)}, {len(fpfs_cldfree_grid)}')

    # Determine Whether to Add Clouds or Not?

    if "H2Oaer" in sol_dict_aer:
        # What if we added Grey Earth-like Clouds?
        
        # Calculate pbot:
        pbot = find_pbot(sol = atm, solaer=sol_dict_aer)

        if pbot is not None:
            print(f'pbot was calculated, there is H2Oaer and a cloud was implemented')
            logpbot = np.log10(pbot)
        
            # Calculate logdp:
            ptop_earth = 0.6
            pbot_earth = 0.7
            logdp = np.log10(pbot_earth) - np.log10(ptop_earth)  
    
            # Default opd (optical depth), w0 (single scattering albedo), g0 (asymmetry parameter)
            start_case.clouds(w0=[0.99], g0=[0.85], 
                              p = [logpbot], dp = [logdp], opd=[10])
            # Cloud spectrum
            df_cld = start_case.spectrum(opacity,full_output=True)
            
            # Average the two spectra - This differs between Calculating Earth Reflected Spectra 
            wno_c, alb_c, fpfs_c, albedo_c = df_cld['wavenumber'],df_cld['albedo'],df_cld['fpfs_reflected'], df_cld['albedo']
            _, alb = jdi.mean_regrid(wno_cldfree, 0.5*alb_cldfree+0.5*albedo_c,R=150)
            wno, fpfs = jdi.mean_regrid(wno_cldfree, 0.5*fpfs_cldfree+0.5*fpfs_c,R=150)

            # Match the length of the clouds array with the length of wno or alb (fpfs is different length)
            clouds = [1] * len(wno)

            
            if outputfile == None:
                return wno, fpfs, alb, np.array(clouds), df_cld, df_cldfree
            else:
                outputfile_name = outputfile
                RSM_outputs = {"wno":wno,
                               "fpfs":fpfs,
                               "alb":alb,
                               "clouds":np.array(clouds),
                               "df_cld":df_cld,
                               "df_cldfree":df_cldfree}
                
                with open(f'RLS_{outputfile_name}.pkl', 'wb') as f:
                    pickle.dump(RSM_outputs, f)
                return wno, fpfs, alb, np.array(clouds), df_cld, df_cldfree

        else:
            print(f'pbot is empty, so no cloud is implemented')
            wno = wno_cldfree_grid.copy()
            alb = alb_cldfree_grid.copy()
            fpfs = fpfs_cldfree_grid.copy()

            # Match the length of the clouds array with the length of wno or alb (fpfs is different length)
            clouds = [0] * len(wno)

            print(f'This is the length of the values I want to save: wno {len(wno)}, alb {len(alb)}, fpfs {len(fpfs)}, clouds {len(clouds)}')

            if outputfile == None:
                return wno, fpfs, alb, np.array(clouds), df_cld, df_cldfree
            else:
                outputfile_name = outputfile
                RSM_outputs = {"wno":wno,
                               "fpfs":fpfs,
                               "alb":alb,
                               "clouds":np.array(clouds),
                               "df_cld":df_cld,
                               "df_cldfree":df_cldfree}
                
                with open(f'RLS_{outputfile_name}.pkl', 'wb') as f:
                    pickle.dump(RSM_outputs, f)
                return wno, fpfs, alb, np.array(clouds), df_cld, df_cldfree

    else:
        print(f'H2Oaer is not in solutions')
        wno = wno_cldfree_grid.copy()
        alb = alb_cldfree_grid.copy()
        fpfs = fpfs_cldfree_grid.copy()
        print(f'For the inputs: {rad_plan}, {planet_metal}, {tint}, {semi_major}, {ctoO}, {Kzz}, {phase_angle}, The length should match: wno - {len(wno)}, alb - {len(alb)}, fpfs - {len(fpfs)}')
        
        # Match the length of the clouds array with the length of wno or alb (fpfs is different length)
        clouds = [0] * len(wno) # This means that there are no clouds

        if outputfile == None:
                return wno, fpfs, alb, np.array(clouds), df_cld, df_cldfree
        else:
            outputfile_name = outputfile
            RSM_outputs = {"wno":wno,
                           "fpfs":fpfs,
                           "alb":alb,
                           "clouds":np.array(clouds),
                           "df_cld":df_cld,
                           "df_cldfree":df_cldfree}
            
            with open(f'RLS_{outputfile_name}.pkl', 'wb') as f:
                pickle.dump(RSM_outputs, f)
                
            return wno, fpfs, alb, np.array(clouds), df_cld, df_cldfree

In [ ]:
rad = 2
metal = 3.125
tint = 400
semi_major = 5
ctoO = 0.01
Kzz = 5
phase_angle = 0
#atmosphere_kwargs = {'exclude_mol': } # What molecules do you want to exclude?

wno, fpfs, alb, clouds, df_cld, df_cldfree = reflected_spectrum_planet_Sun(rad_plan=rad, planet_metal=metal, tint=tint, semi_major=semi_major, ctoO=ctoO, Kzz=Kzz, phase_angle=phase_angle, Photochem_file=photochem_filename , outputfile=f'{rad}_{metal}_{tint}_{semi_major}_{ctoO}_{Kzz}_{phase_angle}')

In [6]:
_photochem_file = 'results/Photochem_1D_updatop_paramext_reducedrad_full_try3.h5'

with h5py.File(_photochem_file, 'r') as _f:
    PHOTOCHEM_INPUTS  = np.array(_f['inputs'])
    PHOTOCHEM_RESULTS = {key: np.array(_f['results'][key]) for key in _f['results'].keys()}

In [7]:
def find_Photochem_match(
    rad_plan=None, log10_planet_metallicity=None, tint=None,
    semi_major=None, ctoO=None, Kzz=None,
    gridvals=photochem_grid.get_gridvals_Photochem()
):
    """
    Find the matching Photochem solution for the given input parameters.
    Searches the in-memory PHOTOCHEM_INPUTS / PHOTOCHEM_RESULTS globals
    (loaded once at worker startup) instead of reopening the HDF5 file.

    Returns (sol_dict, soleq_dict, PT_list, convergence_PC, convergence_TP).
    All are None if no match is found.
    """
    gridvals_metal = [float(s) for s in gridvals[1]]
    gridvals_ctoO  = [float(s) for s in gridvals[4]]

    planet_metallicity = float(log10_planet_metallicity)
    input_list = np.array([rad_plan, planet_metallicity, tint, semi_major, ctoO, Kzz])

    # Search in-memory inputs array — no file I/O
    row_matches = np.all(PHOTOCHEM_INPUTS == input_list, axis=1)
    matching_indicies = np.where(row_matches)[0]

    if matching_indicies.size == 0:
        print(f'No Photochem match found for inputs: {input_list}')
        return None, None, None, None, None

    # Find per-axis indices for multi-dimensional result lookup
    gridvals_dict = {
        'rad_plan':           np.array(gridvals[0]),
        'planet_metallicity': np.array(gridvals_metal),
        'tint':               np.array(gridvals[2]),
        'semi_major':         np.array(gridvals[3]),
        'ctoO':               np.array(gridvals_ctoO),
        'Kzz':                np.array(gridvals[5]),
    }
    ri = np.where(gridvals_dict['rad_plan']           == input_list[0])[0]
    mi = np.where(gridvals_dict['planet_metallicity'] == input_list[1])[0]
    ti = np.where(gridvals_dict['tint']               == input_list[2])[0]
    si = np.where(gridvals_dict['semi_major']         == input_list[3])[0]
    ci = np.where(gridvals_dict['ctoO']               == input_list[4])[0]
    ki = np.where(gridvals_dict['Kzz']                == input_list[5])[0]

    sol_dict   = {}
    soleq_dict = {}
    for key, arr in PHOTOCHEM_RESULTS.items():
        if key.endswith('sol'):
            sol_dict[key]   = arr[ri[0]][mi[0]][ti[0]][si[0]][ci[0]][ki[0]]
        elif key.endswith('soleq'):
            soleq_dict[key] = arr[ri[0]][mi[0]][ti[0]][si[0]][ci[0]][ki[0]]

    sol_dict   = {k.removesuffix('_sol')   if k.endswith('_sol')   else k: v for k, v in sol_dict.items()}
    soleq_dict = {k.removesuffix('_soleq') if k.endswith('_soleq') else k: v for k, v in soleq_dict.items()}

    pressure_values    = PHOTOCHEM_RESULTS['pressure_sol'][ri[0]][mi[0]][ti[0]][si[0]][ci[0]][ki[0]]
    temperature_values = PHOTOCHEM_RESULTS['temperature_sol'][ri[0]][mi[0]][ti[0]][si[0]][ci[0]][ki[0]]
    convergence_PC     = PHOTOCHEM_RESULTS['converged_PC'][ri[0]][mi[0]][ti[0]][si[0]][ci[0]][ki[0]]
    convergence_TP     = PHOTOCHEM_RESULTS['converged_TP'][ri[0]][mi[0]][ti[0]][si[0]][ci[0]][ki[0]]
    PT_list = pressure_values, temperature_values

    print(f'Photochem match found for inputs: {input_list}')
    return sol_dict, soleq_dict, PT_list, convergence_PC, convergence_TP

In [8]:
rad = 2
metal = 3.125
tint = 400
semi_major = 5
ctoO = 0.01
Kzz = 5
phase_angle = 0

sol_dict, soleq_dict, PT_list, convergence_PC, convergence_TP = find_Photochem_match(
    rad_plan=rad, log10_planet_metallicity=metal, tint=tint,
    semi_major=semi_major, ctoO=ctoO, Kzz=Kzz,
    gridvals=photochem_grid.get_gridvals_Photochem()
)

Photochem match found for inputs: [2.000e+00 3.125e+00 4.000e+02 5.000e+00 1.000e-02 5.000e+00]


In [23]:
def reflected_spectrum_planet_Sun(
    rad_plan=None, planet_metal=None, tint=None, semi_major=None,
    ctoO=None, Kzz=None, phase_angle=None,
    atmosphere_kwargs={}, opacity=None
):
    """
    Calculate the reflected spectrum of a mini-Neptune around a Sun-like star,
    using composition from the Photochem grid.

    Photochem data is read from the PHOTOCHEM_INPUTS / PHOTOCHEM_RESULTS globals
    loaded once at worker startup — no file I/O per call.

    Parameters
    ----------
    rad_plan : float
        Planet radius in Earth radii.
    planet_metal : float
        log10(metallicity / Solar).
    tint : float
        Internal temperature (K).
    semi_major : float
        Semi-major axis (AU).
    ctoO : float
        C/O ratio in Solar units.
    Kzz : float
        log10(eddy diffusion coefficient / cm^2 s^-1).
    phase_angle : float
        Phase angle in radians.
    atmosphere_kwargs : dict
        Optional; pass {'exclude_mol': ['O2']} to zero out a species.

    Returns
    -------
    wno, fpfs, alb, clouds : np.ndarray (each length ~150)
    """

    if opacity == None:
        opacity = OPACITY
    else:
        opacity=opacity
    planet_metal = float(planet_metal)

    start_case = jdi.inputs()

    # Planet physical properties
    planet_radius = rad_plan * 6.371e6 * u.m
    planet_mass_earth = Reflected_Spectra_SpecificInputs_Script.mass_from_radius_chen_kipping_2017(R_rearth=rad_plan)
    planet_mass = planet_mass_earth * 5.972e24  # kg
    planet_grav = (const.G.value * planet_mass) / (planet_radius.value**2)  # m/s^2

    # Star parameters (Sun)
    stellar_Teff  = 5778   # K
    stellar_logg  = 4.4    # log10(g / cgs)
    stellar_metal = 0.0    # log10(Z/Solar)
    stellar_radius = 1.0   # Solar radii

    start_case.phase_angle(phase_angle, num_tangle=8, num_gangle=8)

    jupiter_mass   = const.M_jup.value          # kg
    jupiter_radius = 69911e3                    # m
    start_case.gravity(
        gravity=planet_grav, gravity_unit=jdi.u.Unit('m/(s**2)'),
        radius=planet_radius.value / jupiter_radius, radius_unit=jdi.u.Unit('R_jup'),
        mass=planet_mass / jupiter_mass, mass_unit=jdi.u.Unit('M_jup')
    )
    start_case.star(
        opannection=opacity,
        temp=stellar_Teff, logg=stellar_logg,
        semi_major=semi_major, metal=stellar_metal,
        radius=stellar_radius, radius_unit=jdi.u.R_sun,
        semi_major_unit=jdi.u.au
    )

    # Look up Photochem composition (searches in-memory globals, no file I/O)
    sol_dict, soleq_dict, PT_list, convergence_PC, convergence_TP = find_Photochem_match(
        rad_plan=rad_plan, log10_planet_metallicity=planet_metal,
        tint=tint, semi_major=semi_major, ctoO=ctoO, Kzz=Kzz
    )

    if sol_dict is None:
        raise ValueError(
            f'No Photochem match for rad={rad_plan}, metal={planet_metal}, '
            f'tint={tint}, semi_major={semi_major}, ctoO={ctoO}, Kzz={Kzz}'
        )

    atm, sol_dict_aer = Reflected_Spectra_SpecificInputs_Script.make_picaso_atm(sol_dict)

    # Limit atmosphere to layers where temperature < 1000 K.
    # O3 opacity is not valid at temperatures >= 1000 K, so those layers
    # are dropped before passing to PICASO.
    t_threshold = 1000  # K
    temp_arr = np.array(atm['temperature'])
    filtered_temp = temp_arr[temp_arr < t_threshold]
    n_keep = len(filtered_temp)
    atm_filtered = {}
    atm_filtered['temperature'] = filtered_temp
    atm_filtered['pressure'] = np.array(atm['pressure'])[:n_keep]
    for key, value in atm.items():
        if key not in {'pressure', 'temperature'}:
            atm_filtered[key] = np.array(value)[:n_keep]

    df_atmo = jdi.pd.DataFrame(atm_filtered)

    if 'exclude_mol' in atmosphere_kwargs:
        for sp in atmosphere_kwargs['exclude_mol']:
            if sp in df_atmo:
                df_atmo[sp] *= 0

    start_case.atmosphere(df=df_atmo)
    df_cldfree = start_case.spectrum(opacity, calculation='reflected', full_output=True)
    wno_cf = df_cldfree['wavenumber']
    alb_cf = df_cldfree['albedo']
    fpfs_cf = df_cldfree['fpfs_reflected']
    _, alb_cf_grid  = jdi.mean_regrid(wno_cf, alb_cf,  R=150)
    wno_cf_grid, fpfs_cf_grid = jdi.mean_regrid(wno_cf, fpfs_cf, R=150)

    # Optionally add H2O water cloud
    if 'H2Oaer' in sol_dict_aer:
        pbot = Reflected_Spectra_SpecificInputs_Script.find_pbot(sol=atm_filtered, solaer=sol_dict_aer)
        if pbot is not None:
            logpbot = np.log10(pbot)
            ptop_earth, pbot_earth = 0.6, 0.7
            logdp = np.log10(pbot_earth) - np.log10(ptop_earth)
            start_case.clouds(w0=[0.99], g0=[0.85], p=[logpbot], dp=[logdp], opd=[10])
            df_cld = start_case.spectrum(opacity, full_output=True)
            wno_c   = df_cld['wavenumber']
            alb_c   = df_cld['albedo']
            fpfs_c  = df_cld['fpfs_reflected']
            _, alb  = jdi.mean_regrid(wno_cf, 0.5*alb_cf  + 0.5*alb_c,  R=150)
            wno, fpfs = jdi.mean_regrid(wno_cf, 0.5*fpfs_cf + 0.5*fpfs_c, R=150)
            clouds = np.ones(len(wno))
            return wno, fpfs, alb, clouds

    wno   = wno_cf_grid.copy()
    alb   = alb_cf_grid.copy()
    fpfs  = fpfs_cf_grid.copy()
    clouds = np.zeros(len(wno))
    return wno, fpfs, alb, clouds

In [17]:
rad = 2
metal = 3.125
tint = 400
semi_major = 5
ctoO = 0.01
Kzz = 5
phase_angle = 0

current_directory = Path.cwd()
opacity_file_path = "Installation&Setup_Instructions/picasofiles/reference/opacities/opacities_photochem_0.1_250.0_R15000.db"  
opacity_path=os.path.join(current_directory, opacity_file_path)
print(opacity_path)
opacity_loaded = jdi.opannection(filename_db=opacity_path, wave_range=[0.1,2.5])

/mmfs1/gscratch/stf/elizap/MiniNeptuneGrid26_PostBac/Installation&Setup_Instructions/picasofiles/reference/opacities/opacities_photochem_0.1_250.0_R15000.db


In [24]:
wno, fpfs, alb, clouds = reflected_spectrum_planet_Sun(
    rad_plan=rad, planet_metal=metal, tint=tint, semi_major=semi_major,
    ctoO=ctoO, Kzz=Kzz, phase_angle=phase_angle,
    atmosphere_kwargs={}, opacity = opacity_loaded)

Photochem match found for inputs: [2.000e+00 3.125e+00 4.000e+02 5.000e+00 1.000e-02 5.000e+00]


In [9]:
wno, fpfs, alb, clouds = Reflected_Spectra_SpecificInputs_Script.reflected_spectrum_planet_Sun(
    rad_plan=rad, planet_metal=metal, tint=tint, semi_major=semi_major,
    ctoO=ctoO, Kzz=Kzz, phase_angle=phase_angle,
    atmosphere_kwargs={}, opacity = opacity_loaded)

/mmfs1/gscratch/stf/elizap/MiniNeptuneGrid26_PostBac/Installation&Setup_Instructions/picasofiles/reference/opacities/opacities_photochem_0.1_250.0_R15000.db


AxisError: axis 1 is out of bounds for array of dimension 1

In [10]:
print(type(PHOTOCHEM_INPUTS))

<class 'numpy.ndarray'>


In [13]:
input_list = [rad, metal, tint, ctoO, Kzz, phase_angle]
np.all(PHOTOCHEM_INPUTS == input_list, axis = 1)

array([False, False, False, ..., False, False, False])

In [25]:
import h5py
import numpy as np

filename = 'results/ReflectedSpectra_SpecificInputsModernEarth_fv.h5'

with h5py.File(filename, 'r') as f:

    # Input parameters for every case — shape (N, 7)
    # Columns: [rad, metal, tint, semi_major, ctoO, kzz, phase_angle]
    print(f.keys())
    print(list(f['results']['status']))
    inputs = np.array(f['inputs'])

    # Which cases finished successfully — shape (N,)
    completed = np.array(f['completed'])

    # Spectral results — each shape (N, ~150)
    wno    = np.array(f['results']['wno'])
    fpfs   = np.array(f['results']['fpfs'])
    albedo = np.array(f['results']['albedo'])
    clouds = np.array(f['results']['clouds'])
    status = np.array(f['results']['status'])   # byte strings, e.g. b'ok'

print(f"Total cases:     {len(inputs)}")
print(f"Completed cases: {completed.sum()}")
print(f"wno shape:       {wno.shape}")   # (N, ~150)


FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = 'results/ReflectedSpectra_SpecificInputsModernEarth_fv.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)